# Knee Abnormality ViT — Colab Training

**Workflow:**
1. Edit code in VS Code → push to GitHub
2. Run this notebook on Colab GPU
3. Results auto-saved to Google Drive

**First time only:** set your GitHub repo URL in the cell below.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/YOUR_USERNAME/knee-abnormality-vit.git"  # <-- set this once
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/knee-abnormality-vit"
# ────────────────────────────────────────────────────────────────────────────

## Step 1 — Mount Google Drive & clone/pull repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = "/content/knee-abnormality-vit"

if os.path.exists(REPO_DIR):
    # Pull latest code you pushed from VS Code
    !git -C {REPO_DIR} pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## Step 2 — Verify GPU

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3 — Link dataset from Drive

In [ ]:
import os
from pathlib import Path

# Expected layout in Drive:
#   MyDrive/knee-abnormality-vit/data/
#     train/normal/  train/acl/  train/meniscus/
#     val/normal/    val/acl/    val/meniscus/
#     test/normal/   test/acl/   test/meniscus/

DRIVE_DATA = Path(DRIVE_PROJECT_DIR) / "data"
assert DRIVE_DATA.exists(), f"Put your dataset at {DRIVE_DATA}"
print("Dataset found:", DRIVE_DATA)
for split in ("train", "val", "test"):
    n = sum(1 for _ in (DRIVE_DATA / split).rglob("*.png")) + \
        sum(1 for _ in (DRIVE_DATA / split).rglob("*.jpg"))
    print(f"  {split}: {n} images")

## Step 4 — Train

In [ ]:
import sys, yaml
from pathlib import Path

sys.path.insert(0, str(Path(REPO_DIR)))

from src.dataset import get_dataloaders
from src.model import build_model
from src.trainer import train

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

output_dir = Path(DRIVE_PROJECT_DIR) / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

loaders, classes = get_dataloaders(str(DRIVE_DATA), cfg)
print("Classes:", classes)

cfg["model"]["num_classes"] = len(classes)
model = build_model(cfg).to(device)

history = train(model, loaders, cfg, output_dir, device)
print("\nBest checkpoint saved to:", output_dir / "best_model.pth")

## Step 5 — Evaluate on test set

In [ ]:
import torch
from src.trainer import evaluate
import torch.nn as nn

model.load_state_dict(torch.load(output_dir / "best_model.pth", map_location=device))
test_loss, test_acc = evaluate(model, loaders["test"], nn.CrossEntropyLoss(), device)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

## Step 6 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="val")
ax1.set_title("Loss"); ax1.legend()
ax2.plot(history["train_acc"], label="train")
ax2.plot(history["val_acc"], label="val")
ax2.set_title("Accuracy"); ax2.legend()
plt.tight_layout()
plt.savefig(output_dir / "training_curves.png")
plt.show()